In [1]:
!pip -q install -U datasets huggingface_hub

import os
from huggingface_hub import login, HfApi

# Put your token here once for the runtime
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

# Log in this Python process
login(token=os.environ["HF_TOKEN"])

# Sanity check: who am I?
api = HfApi()
print("Whoami:", api.whoami(os.environ["HF_TOKEN"])["name"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 61.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Whoami: MarcoPolo4


In [2]:
# %% Train BTRM on LitBench-Train (HF) using Meta-Llama; eval on LitBench-Test-Enhanced; push to Hugging Face Hub
!pip -q install -U "transformers>=4.38.0" "datasets>=2.18.0" "accelerate>=0.28.0" "trl>=0.9.6" sentencepiece huggingface_hub

import os, json, math, time, random, inspect, traceback, textwrap
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
from trl import RewardTrainer, RewardConfig
from trl.trainer.utils import RewardDataCollatorWithPadding
from huggingface_hub import login, create_repo, upload_folder, upload_file, HfApi

# ----------------------------- config -----------------------------
class Cfg:
    seed = 42

    # HF auth & repo info (EDIT THESE)
    HF_TOKEN = os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"         # set once: os.environ["HF_TOKEN"] = "hf_xxx"
    HF_USERNAME  = os.environ.get("MarcoPolo4", "").strip()      # e.g., "your-hf-handle"
    MODEL_REPO   = os.environ.get("LitBench-BTRM", "btrm-litbench-llama32-1b").strip()
    CODE_REPO    = os.environ.get("BTRM-training",  "btrm-litbench-training-code").strip()

    # backbone (switch to Meta-Llama)
    base_model = "meta-llama/Llama-3.2-1B"   # requires access + HF token

    # data sizing
    train_target = 30000
    val_target   = 2000
    max_length   = 512

    # optimization
    num_train_epochs = 1
    per_device_train_batch_size = 2
    per_device_eval_batch_size  = 2
    gradient_accumulation_steps = 8
    learning_rate   = 1e-5
    weight_decay    = 0.1
    max_grad_norm   = 1.0
    warmup_ratio    = 0.10
    logging_steps   = 50
    eval_steps      = 200
    save_steps      = 1000

    # runtime / output
    output_dir = "bt_llama32_final_assistant_litbenchHF_meta"
    report_to  = []  # no W&B

# seeds & math dtypes
set_seed(Cfg.seed)
torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA:", torch.cuda.is_available(), "| Device:", device)

# ----------------------------- load datasets -----------------------------
print("Loading SAA-Lab/LitBench-Train (for TRAIN)…")
ds_train = load_dataset("SAA-Lab/LitBench-Train", split="train", token=Cfg.HF_TOKEN)

print("Loading SAA-Lab/LitBench-Test-Enhanced (for TEST)…")
# This dataset only exposes a 'train' split; use it as your TEST set
ds_test_ext = load_dataset("SAA-Lab/LitBench-Test-Enhanced", split="train", token=Cfg.HF_TOKEN)

def extract_pairs(split):
    def get(ex, keys):
        for k in keys:
            if k in ex and ex[k] is not None:
                return str(ex[k])
        return ""
    chosen = [get(ex, ["chosen_story","chosen","chosen_text"]).strip() for ex in split]
    reject = [get(ex, ["rejected_story","rejected","rejected_text"]).strip() for ex in split]
    keep   = [(c, r) for c, r in zip(chosen, reject) if c and r and c != r]
    ch = [c for c,_ in keep]; rj = [r for _,r in keep]
    return ch, rj

tr_ch_all, tr_rj_all = extract_pairs(ds_train)
te_ch_ext, te_rj_ext = extract_pairs(ds_test_ext)

print(f"TRAIN pairs: {len(tr_ch_all)} | TEST-ENH pairs: {len(te_ch_ext)}")

# carve a small VAL out of train
N = len(tr_ch_all)
idx = list(range(N)); random.Random(Cfg.seed).shuffle(idx)
n_val = min(Cfg.val_target or 0, max(100, N//20))
n_train = min(Cfg.train_target or N, N - n_val)
vl_idx = idx[:n_val]; tr_idx = idx[n_val:n_val+n_train]
tr_ch_txt = [tr_ch_all[i] for i in tr_idx]; tr_rj_txt = [tr_rj_all[i] for i in tr_idx]
vl_ch_txt = [tr_ch_all[i] for i in vl_idx]; vl_rj_txt = [tr_rj_all[i] for i in vl_idx]

print(f"Final sizes -> TRAIN={len(tr_ch_txt)} | VAL={len(vl_ch_txt)} | EXT-TEST={len(te_ch_ext)}")

# ----------------------------- tokenizer -----------------------------
tok = AutoTokenizer.from_pretrained(Cfg.base_model, token=Cfg.HF_TOKEN, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id
print("Pad token:", tok.pad_token, "| id:", tok.pad_token_id)

# ----------------------------- tokenization -----------------------------
def tok_pair_batch_texts(chosen_list, rejected_list, tokenizer, max_length):
    t_ch = tokenizer(chosen_list,   truncation=True, padding=False, max_length=max_length)
    t_rj = tokenizer(rejected_list, truncation=True, padding=False, max_length=max_length)
    return Dataset.from_dict({
        "input_ids_chosen":      t_ch["input_ids"],
        "attention_mask_chosen": t_ch["attention_mask"],
        "input_ids_rejected":      t_rj["input_ids"],
        "attention_mask_rejected": t_rj["attention_mask"],
    })

print("Tokenizing…")
tok_train = tok_pair_batch_texts(tr_ch_txt, tr_rj_txt, tok, Cfg.max_length)
tok_val   = tok_pair_batch_texts(vl_ch_txt, vl_rj_txt, tok, Cfg.max_length) if len(vl_ch_txt) else None

# ----------------------------- model (full finetune) -----------------------------
print("Loading model…")
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
model = AutoModelForSequenceClassification.from_pretrained(
    Cfg.base_model,
    token=Cfg.HF_TOKEN,                 # <-- gated model access
    num_labels=1,
    torch_dtype=torch.bfloat16 if bf16_ok else None,
    device_map="auto",
)
model.config.pad_token_id = tok.pad_token_id
for p in model.parameters(): p.requires_grad = True
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

# ----------------------------- TRL trainer -----------------------------
collator = RewardDataCollatorWithPadding(tokenizer=tok, padding=True)
args = RewardConfig(
    output_dir=Cfg.output_dir,
    num_train_epochs=Cfg.num_train_epochs,
    per_device_train_batch_size=Cfg.per_device_train_batch_size,
    per_device_eval_batch_size=Cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=Cfg.gradient_accumulation_steps,
    learning_rate=Cfg.learning_rate,
    weight_decay=Cfg.weight_decay,
    max_grad_norm=Cfg.max_grad_norm,
    warmup_ratio=Cfg.warmup_ratio,
    logging_steps=Cfg.logging_steps,
    save_strategy="steps",
    save_steps=Cfg.save_steps,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=Cfg.report_to,
    bf16=bool(bf16_ok),
    disable_dropout=True,
)

trainer = RewardTrainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
)
trainer.tokenizer = tok

# ----------------------------- pairwise accuracy helper -----------------------------
@torch.no_grad()
def pairwise_acc_texts(model, tok, chosen, rejected, max_length=Cfg.max_length, bs=32):
    model.eval()
    ok = tot = 0
    for i in range(0, len(chosen), bs):
        e1 = tok(chosen[i:i+bs], padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        e2 = tok(rejected[i:i+bs], padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        e1 = {k:v.to(model.device) for k,v in e1.items()}
        e2 = {k:v.to(model.device) for k,v in e2.items()}
        s1 = model(**e1).logits.squeeze(-1); s2 = model(**e2).logits.squeeze(-1)
        ok += (s1 > s2).sum().item(); tot += s1.numel()
    return ok / max(1, tot)

print("Pre-train EXT-TEST acc:", f"{pairwise_acc_texts(model, tok, te_ch_ext, te_rj_ext):.3f}")

# ----------------------------- train -----------------------------
print("Training…")
trainer.train()
trainer.save_model(); tok.save_pretrained(Cfg.output_dir)
print("Saved to:", Cfg.output_dir)

# ----------------------------- post-train metrics -----------------------------
print("Post-train TRAIN acc:", f"{pairwise_acc_texts(model, tok, tr_ch_txt, tr_rj_txt):.3f}")
print("Post-train EXT-TEST acc:", f"{pairwise_acc_texts(model, tok, te_ch_ext, te_rj_ext):.3f}")

# ----------------------------- artifacts (pairs + run config) -----------------------------
art_dir = os.path.join(Cfg.output_dir, "artifacts")
os.makedirs(art_dir, exist_ok=True)
def _dump_jsonl_pairs(path, chosen, rejected):
    with open(path, "w", encoding="utf-8") as f:
        for c, r in zip(chosen, rejected):
            f.write(json.dumps({"chosen": c, "rejected": r}, ensure_ascii=False) + "\n")

with open(os.path.join(art_dir, "run_config.json"), "w") as f:
    json.dump({
        "dataset_train": "SAA-Lab/LitBench-Train",
        "dataset_test": "SAA-Lab/LitBench-Test-Enhanced",
        "model": Cfg.base_model,
        "max_length": Cfg.max_length,
        "epochs": Cfg.num_train_epochs,
        "per_device_train_bs": Cfg.per_device_train_batch_size,
        "per_device_eval_bs": Cfg.per_device_eval_batch_size,
        "grad_accum": Cfg.gradient_accumulation_steps,
        "learning_rate": Cfg.learning_rate,
        "weight_decay": Cfg.weight_decay,
        "warmup_ratio": Cfg.warmup_ratio,
        "train_pairs": len(tr_ch_txt),
        "val_pairs": len(vl_ch_txt),
        "ext_test_pairs": len(te_ch_ext),
    }, f, indent=2)

_dump_jsonl_pairs(os.path.join(art_dir, "train_pairs.jsonl"), tr_ch_txt, tr_rj_txt)
_dump_jsonl_pairs(os.path.join(art_dir, "val_pairs.jsonl"),   vl_ch_txt, vl_rj_txt)
_dump_jsonl_pairs(os.path.join(art_dir, "ext_test_pairs.jsonl"), te_ch_ext, te_rj_ext)
print("Wrote artifacts to:", art_dir)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 35.5 MB/s eta 0:00:00
CUDA: True | Device: cuda
Loading SAA-Lab/LitBench-Train (for TRAIN)…


README.md:   0%|          | 0.00/725 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/173M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43827 [00:00<?, ? examples/s]

Loading SAA-Lab/LitBench-Test-Enhanced (for TEST)…


...

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The model is already on multiple devices. Skipping the move to device specified in `args`.
Trainer.tokenizer is now deprecated. You should use `Trainer.processing_class = processing_class` instead.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Pre-train EXT-TEST acc: 0.497
Training…


Step,Training Loss
50,1.067800
100,1.009000
150,0.950300
200,0.818200
250,0.754600
300,0.687800
350,0.660700
400,0.626300
450,0.595300


Step,Training Loss
50,1.067800
100,1.009000
150,0.950300
200,0.818200
250,0.754600
300,0.687800
350,0.660700
400,0.626300
450,0.595300
500,0.624500


Saved to: bt_llama32_final_assistant_litbenchHF_meta
Post-train TRAIN acc: 0.794
Post-train EXT-TEST acc: 0.734
Wrote artifacts to: bt_llama32_final_assistant_litbenchHF_meta/artifacts


In [3]:
!pip -q install -U "transformers==4.43.3" "datasets>=2.18.0,<3.0" "accelerate>=0.28.0,<0.34" sentencepiece einops numpy>=2.0.0

import os, math, random, itertools, numpy as np, torch, torch.nn as nn, torch.optim as optim
from typing import List, Dict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed

# --------------------- CONFIG ---------------------
class Cfg:
    seed = 42
    HF_TOKEN   = os.environ.get("HF_TOKEN", "").strip()
    BASE_MODEL = "meta-llama/Llama-3.2-1B"
    BTRM_DIR   = "bt_llama32_final_assistant_litbenchHF_meta"
    SAVE_DIR   = f"{BTRM_DIR}/sae_bt_pure"

    max_length = 512
    bs_encode  = 16
    LAYER_INDEX = -1
    TOKEN_RULE  = "last_non_pad"

    sample_cap_train = 10000
    sample_cap_test  = 4000

    # SAE size/sparsity
    active_frac = 0.05   # ~5% active codes
    sae_epochs  = 3
    sae_bs      = 128
    sae_lr      = 1e-3

    # BT head
    bt_lr=1e-2; bt_wd=1e-4; bt_epochs=10; bt_bs=4096

set_seed(Cfg.seed)
os.makedirs(Cfg.SAVE_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
print("Device:", device, "| bf16:", bf16_ok)

# helpers
def _get(ex, keys):
    for k in keys:
        if k in ex and ex[k] is not None:
            return str(ex[k])
    return ""

def get_pairs(ds):
    ch = [_get(ex, ["chosen_story","chosen","chosen_text"]).strip() for ex in ds]
    rj = [_get(ex, ["rejected_story","rejected","rejected_text"]).strip() for ex in ds]
    return [{"chosen": c, "rejected": r} for c, r in zip(ch, rj) if c and r and c != r]

def pick_texts(pairs, cap=None):
    texts = list(itertools.chain.from_iterable([[p["chosen"], p["rejected"]] for p in pairs]))
    if cap and len(texts) > cap:
        random.Random(Cfg.seed).shuffle(texts); texts = texts[:cap]
    return texts

def last_non_pad_index(attn_mask: torch.Tensor) -> torch.Tensor:
    idx = attn_mask.sum(dim=1) - 1
    return torch.clamp(idx, min=0)

def eos_if_present_index(input_ids: torch.Tensor, eos_id: int, attn_mask: torch.Tensor) -> torch.Tensor:
    B, T = input_ids.shape
    has_eos = (input_ids == eos_id)
    idx_eos = torch.where(has_eos, torch.arange(T, device=input_ids.device)[None, :], torch.tensor(-1, device=input_ids.device))
    last_eos = idx_eos.max(dim=1).values
    fallback = last_non_pad_index(attn_mask)
    return torch.where(last_eos >= 0, last_eos, fallback)

def select_token(hidden_states, input_ids, attn_mask, tok):
    idx = eos_if_present_index(input_ids, tok.eos_token_id, attn_mask) if Cfg.TOKEN_RULE=="eos_if_present" else last_non_pad_index(attn_mask)
    B = hidden_states.size(0)
    return hidden_states[torch.arange(B, device=hidden_states.device), idx, :]

@torch.no_grad()
def encode_texts(model, tokenizer, texts):
    vecs = []
    for i in range(0, len(texts), Cfg.bs_encode):
        batch = texts[i:i+Cfg.bs_encode]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=Cfg.max_length, return_tensors="pt").to(device)
        hs = model(**enc).hidden_states[Cfg.LAYER_INDEX]
        sel = select_token(hs, enc["input_ids"], enc["attention_mask"], tokenizer).float()
        vecs.append(sel.cpu())
    return torch.cat(vecs, 0) if vecs else torch.zeros(0, hidden_dim)

# load BASE and BTRM
tok_btrm = AutoTokenizer.from_pretrained(Cfg.BTRM_DIR, use_fast=True)
if tok_btrm.pad_token is None:
    tok_btrm.pad_token = tok_btrm.eos_token
    tok_btrm.pad_token_id = tok_btrm.eos_token_id

btrm_model = AutoModelForSequenceClassification.from_pretrained(
    Cfg.BTRM_DIR, num_labels=1, torch_dtype=torch.bfloat16 if bf16_ok else None
).to(device)
btrm_model.config.pad_token_id = tok_btrm.pad_token_id
btrm_model.eval(); btrm_model.config.output_hidden_states = True

with torch.no_grad():
    dmy = tok_btrm("hello", return_tensors="pt").to(device)
    hidden_dim = btrm_model(**dmy).hidden_states[Cfg.LAYER_INDEX].shape[-1]
print("BTRM hidden dim:", hidden_dim)

# also load BASE for raw acc later
tok_base = AutoTokenizer.from_pretrained(Cfg.BASE_MODEL, token=Cfg.HF_TOKEN, use_fast=True)
if tok_base.pad_token is None:
    tok_base.pad_token = tok_base.eos_token
    tok_base.pad_token_id = tok_base.eos_token_id
base_model = AutoModelForSequenceClassification.from_pretrained(
    Cfg.BASE_MODEL, token=Cfg.HF_TOKEN, num_labels=1, torch_dtype=torch.bfloat16 if bf16_ok else None
).to(device)
base_model.config.pad_token_id = tok_base.pad_token_id
base_model.eval()

# data
pairs_train = get_pairs(load_dataset("SAA-Lab/LitBench-Train", split="train", token=Cfg.HF_TOKEN))
pairs_test  = get_pairs(load_dataset("SAA-Lab/LitBench-Test-Enhanced", split="train", token=Cfg.HF_TOKEN))
texts_train = pick_texts(pairs_train, Cfg.sample_cap_train)
texts_test  = pick_texts(pairs_test,  Cfg.sample_cap_test)
print(f"Texts → TRAIN={len(texts_train)} | TEST={len(texts_test)}")

# BTRM embeddings
acts_train = encode_texts(btrm_model, tok_btrm, texts_train)   # [N,D]
acts_test  = encode_texts(btrm_model, tok_btrm, texts_test)    # [N,D]

# batch top-k
class SAE(nn.Module):
    def __init__(self, d, m, tie=True):
        super().__init__()
        self.encoder = nn.Linear(d, m, bias=False)
        self.bias_e  = nn.Parameter(torch.zeros(m))
        self.decoder = None if tie else nn.Linear(m, d, bias=False)
        nn.init.kaiming_uniform_(self.encoder.weight, a=math.sqrt(5))
    def forward(self, x, K):
        z = torch.relu(self.encoder(x) + self.bias_e)
        if K and K > 0:
            B, m = z.shape
            nnz = min(B*K, B*m)
            thr = torch.topk(z.reshape(-1), k=nnz, sorted=True).values[-1]
            z = z * (z >= thr)
        x_hat = torch.matmul(z, self.encoder.weight) if self.decoder is None else self.decoder(z)
        return z, x_hat

M = 4 * hidden_dim
K = max(1, int(Cfg.active_frac * M))
print("SAE M:", M, "| K:", K)

def train_sae(acts, tag):
    sae = SAE(acts.shape[1], M, tie=True).to(device)
    opt = optim.AdamW(sae.parameters(), lr=Cfg.sae_lr)
    idx = np.arange(acts.shape[0])
    for ep in range(1, Cfg.sae_epochs+1):
        np.random.default_rng(Cfg.seed+ep).shuffle(idx)
        ep_loss = 0.0
        for i in range(0, len(idx), Cfg.sae_bs):
            b = torch.from_numpy(acts[idx[i:i+Cfg.sae_bs]].numpy()).to(device)
            z, xh = sae(b, K=K)
            loss = ((xh - b)**2).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            ep_loss += loss.item() * b.size(0)
        print(f"[{tag}] epoch {ep}/{Cfg.sae_epochs} loss={ep_loss/len(idx):.4f}")
    return sae

sae = train_sae(acts_train, "BTRM-SAE")

# codes for test pairs
@torch.no_grad()
def sae_codes(sae, texts):
    Z = []
    for i in range(0, len(texts), Cfg.bs_encode):
        batch = texts[i:i+Cfg.bs_encode]
        enc = tok_btrm(batch, padding=True, truncation=True, max_length=Cfg.max_length, return_tensors="pt").to(device)
        hs  = btrm_model(**enc).hidden_states[Cfg.LAYER_INDEX]
        sel = select_token(hs, enc["input_ids"], enc["attention_mask"], tok_btrm).float()
        z, _ = sae(sel, K=K)
        Z.append(z.cpu())
    return torch.cat(Z, 0).numpy() if Z else np.zeros((0, M), dtype=np.float32)

test_ch = [p["chosen"] for p in pairs_test][:Cfg.sample_cap_test]
test_rj = [p["rejected"] for p in pairs_test][:Cfg.sample_cap_test]
X_ch = sae_codes(sae, test_ch); X_rj = sae_codes(sae, test_rj)

# standardize
mu, sig = X_ch.mean(0), X_ch.std(0) + 1e-8
def stdz(A): return (A - mu) / sig
X_ch, X_rj = stdz(X_ch), stdz(X_rj)

# BT Linear
class BTLinear(nn.Module):
    def __init__(self, d): super().__init__(); self.w = nn.Parameter(torch.zeros(d))
    def score(self, X): return X @ self.w
def bt_loss(s_ch, s_rj): return torch.nn.functional.softplus(-(s_ch - s_rj)).mean()
@torch.no_grad()
def bt_pair_metrics(s_ch_np, s_rj_np):
    margin = s_ch_np - s_rj_np
    return {"acc": float((margin > 0).mean()),
            "logloss": float(np.logaddexp(0.0, -margin).mean())}
def train_bt_linear(Z_ch, Z_rj):
    torch.manual_seed(Cfg.seed)
    d = Z_ch.shape[1]
    model = BTLinear(d).to(device)
    opt = optim.AdamW(model.parameters(), lr=Cfg.bt_lr, weight_decay=Cfg.bt_wd)
    Zc = torch.tensor(Z_ch, dtype=torch.float32); Zr = torch.tensor(Z_rj, dtype=torch.float32)
    n  = Zc.size(0)
    model.train()
    for ep in range(Cfg.bt_epochs):
        perm = torch.randperm(n); ep_loss = 0.0
        for i in range(0, n, Cfg.bt_bs):
            idx = perm[i:i+Cfg.bt_bs]
            zc = Zc[idx].to(device); zr = Zr[idx].to(device)
            loss = bt_loss(model.score(zc), model.score(zr))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            ep_loss += loss.item() * zc.size(0)
        print(f"[BT] epoch {ep+1}/{Cfg.bt_epochs} loss={ep_loss/n:.4f}")
    model.eval(); return model
@torch.no_grad()
def bt_scores(model, Z): return model.score(torch.tensor(Z, dtype=torch.float32, device=device)).cpu().numpy()

bt = train_bt_linear(X_ch, X_rj)
metrics = bt_pair_metrics(bt_scores(bt, X_ch), bt_scores(bt, X_rj))
print("\n=== BT-LINEAR (BTRM+BatchTopK-SAE) on LitBench-Test-Enhanced ===")
print(metrics)
print("BT Accuracy:", metrics["acc"])

# baselines
def _scalar_score(logits: torch.Tensor) -> torch.Tensor:
    if logits.ndim == 0: return logits
    if logits.size(-1) == 1: return logits.squeeze(-1)
    if logits.size(-1) == 2: return logits[...,0] - logits[...,1]
    return logits.mean(-1)

@torch.no_grad()
def raw_pair_acc(model, tokenizer, pairs, max_len=Cfg.max_length, bs=32) -> float:
    ok = tot = 0
    for i in range(0, len(pairs), bs):
        ch = [p["chosen"] for p in pairs[i:i+bs]]
        rj = [p["rejected"] for p in pairs[i:i+bs]]
        e1 = tokenizer(ch, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        e2 = tokenizer(rj, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        e1 = {k:v.to(model.device) for k,v in e1.items()}
        e2 = {k:v.to(model.device) for k,v in e2.items()}
        s1 = _scalar_score(model(**e1).logits)
        s2 = _scalar_score(model(**e2).logits)
        ok += (s1 > s2).sum().item(); tot += s1.numel()
    return ok / max(1, tot)

print("\n=== RAW pairwise accuracy (logits) on LitBench-Test-Enhanced ===")
acc_base = raw_pair_acc(base_model, tok_base, pairs_test)
acc_btrm = raw_pair_acc(btrm_model, tok_btrm, pairs_test)
print(f"BASE  ({Cfg.BASE_MODEL}) : {acc_base:.4f}")
print(f"BTRM  ({Cfg.BTRM_DIR}) : {acc_btrm:.4f}")
print(f"BT (SAE codes): {metrics['acc']:.4f}")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.23.1 requires accelerate>=1.4.0, but you have accelerate 0.33.0 which is incompatible.
trl 0.23.1 requires datasets>=3.0.0, but you have datasets 2.21.0 which is incompatible.
trl 0.23.1 requires transformers>=4.56.1, but you have transformers 4.43.3 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Texts → TRAIN=10000 | TEST=4000
SAE M: 8192 | K: 409
[BTRM-SAE] epoch 1/3 loss=0.6129
[BTRM-SAE] epoch 2/3 loss=0.1894
[BTRM-SAE] epoch 3/3 loss=0.1526
[BT] epoch 1/10 loss=0.6931
[BT] epoch 2/10 loss=0.6095
[BT] epoch 3/10 loss=0.5244
[BT] epoch 4/10 loss=0.5410
[BT] epoch 5/10 loss=0.5252
[BT] epoch 6/10 loss=0.4915
[BT] epoch 7/10 loss=0.4821
[BT] epoch 8/10 loss=0.4758
[BT] epoch 9/10 loss=0.4613
[BT] epoch 10/10 loss=0.4393

=== BT-LINEAR (BTRM+BatchTopK-SAE) on LitBench-Test-Enhanced ===
{'acc': 0.7980524978831499, 'logloss': 0.42604967951774597}
BT Accuracy: 0.7980524978831499

=== RAW pairwise accuracy (logits) on LitBench-Test-Enhanced ===
BASE  (meta-llama/Llama-3.2-1B) : 0.4970
BTRM  (bt_llama32_final_assistant_litbenchHF_meta) : 0.7337
BT (SAE codes): 0.7981
